In [ ]:
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import torch
from interactive_seg_backend import (
    FeatureConfig,
    TrainingConfig,
    concat_feats,
    featurise,
    train_and_apply,
)
from interactive_seg_backend.file_handling import load_labels
from PIL import Image
from skimage.transform import resize

from dinosaw.utils import add_custom_font, do_2D_pca, get_features
from dinosaw.wrappers import MODEL_NAMES, ModelTypes, get_models

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False

In [ ]:
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dvt_dinov2_s', 'alibi_coco_dinov2_s', 'dinov3_s+', 'alibi_dinov3_s+_norm_wrap_ms')
models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

f_cfg = FeatureConfig()

tr_cfg = TrainingConfig(feature_config=f_cfg, CRF=True, classifier='xgb', CRF_params={"label_confidence": 0.6,
    "sxy_g": [1, 1],
    "sxy_b": [30, 30],
    "s_rgb": [13, 13, 13],
    "compat_g": 10,
    "compat_b": 10,
    "n_infer": 10}, classifier_params={"class_weight": "balanced", "max_depth": 8}) 

In [ ]:
ds_folder = '../paper_figures_pdf/data/trainable_seg'

AllowedImages = Literal['anode', 'curtains', 'bimodal', 'NMC', 'steel', 'rome', 'butterflies', 'au_foil']


side_lengths: dict[AllowedImages, int] = {
    'au_foil': 737,
    'anode': 756,
    'curtains': 756,
    'rome': 756,
    'butterflies': 756,
    'bimodal': 839,
    'NMC': 602,
    'steel': 602,
    'pre_kint': 1024,
}


image_fnames: list[AllowedImages] = ['rome', 'curtains', 'anode']
# image_fnames: list[AllowedImages] = ['curtains', 'anode', 'pre_kint']
images: list[Image.Image] = []
labels: list[np.ndarray] = []
for image_fname in image_fnames:
    img = Image.open(f"{ds_folder}/{image_fname}.png").convert("RGB")
    shortest = min(img.size)
    L = side_lengths[image_fname]
    sf = L / shortest
    new_size = (int(img.size[0] * sf), int(img.size[1] * sf))
    img = img.resize(new_size, resample=Image.BILINEAR)
    images.append(img)

    label = load_labels(f"{ds_folder}/{image_fname}_biased_labels.tiff")
    labels.append(label)

In [ ]:
features: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}
red_features: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}


for key, model in models.items():
    for img in images:

        classical = featurise(np.array(img.convert('L')), tr_cfg)

        if key == 'classical':
            features[key].append(classical)
            red_features[key].append(classical)
            continue

        feats = get_features(model, img, False)
        reduced = do_2D_pca(feats, 9, pre_norm='std', post_norm='minmax')
        reduced_hr = resize(reduced, (img.height, img.width), order=1, mode='reflect', anti_aliasing=True)

        combined_feats = concat_feats(reduced_hr, classical)
        features[key].append(combined_feats)

        red_features[key].append(reduced_hr)

In [ ]:
preds: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}

for key, model in models.items():
    for i, feat in enumerate(features[key]):
        image = np.array(images[i])
        label = labels[i]

        pred, _, _ = train_and_apply(feat, label, tr_cfg, image=image)
        preds[key].append(pred)

In [ ]:
from PIL.ImageColor import getcolor
from skimage.color import label2rgb

COLOURS = [
    "#648FFF",
    "#785EF0",
    "#DC267F",
    "#FE6100",
    "#FFB000"
]
COLORS = [[v / 255.0 for v in getcolor(c, "RGB")] for c in COLOURS]



# add_custom_font('resources/fonts', 'Grotesk')

def hide_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])



def apply_labels_as_overlay(labels: np.ndarray, img: Image.Image, colors: list, alpha: float=1.0) -> Image.Image:
    labels_unsqueezed = np.expand_dims(labels, -1)

    overlay = label2rgb(labels, colors=colors[1:], kind='overlay', bg_label=0, image_alpha=1, alpha=alpha)
    out = np.where(labels_unsqueezed, overlay * 255, np.array(img)).astype(np.uint8)
    img_with_labels = Image.fromarray(out)
    return img_with_labels

In [ ]:
# %%capture
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')
# W, H = 7, 2.3 * 3.25
W, H = 7.5, 2.3 * 4

N_ROWS = len(enabled_models) + 1
N_COLS = len(image_fnames)
fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(W, H))

titles = ["Rome", "Cathode", "Anode",]

for y in range(N_ROWS):
    for x in range(N_COLS):
        if y == 0:
            axs[y, 0].set_ylabel("Image + labels")
            axs[y, x].set_title(titles[x])
            arr = apply_labels_as_overlay(labels[x], images[x], COLORS, alpha=1)
        else:
            key = enabled_models[y-1]
            # model_name = MODEL_NAMES[key]
            model_name = key
            model_name = model_name.replace('(COCO)', '')
            weight = 700 if 'alibi' in key.lower() else 500
            axs[y, 0].set_ylabel(model_name, weight=weight)
            arr = preds[key][x]
            arr = label2rgb(arr + 1, colors=COLORS[1:], bg_label=0)
        axs[y, x].imshow(arr, aspect='auto', interpolation='nearest')
        hide_axis(axs[y, x])

SAVE = False
if SAVE:
    plt.savefig('saved/09.pdf', dpi=300, bbox_inches='tight')
    plt.close()
